In [35]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
import warnings

In [36]:
warnings.filterwarnings("ignore")

In [37]:
from pathlib import Path

FILE_1 = "global_bleaching_environmental.csv"
FILE_2 = "realistic_ocean_climate_dataset.csv"
OUTPUT_DIR = Path.cwd() / "data" if (Path.cwd() / "data").exists() else Path.cwd().parent / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "bleaching_model_ready.csv"

print("="*50)
print("🚀 เริ่มต้นกระบวนการ Data Cleaning (แบบ Step-by-Step)")
print("="*50)

🚀 เริ่มต้นกระบวนการ Data Cleaning (แบบ Step-by-Step)


In [38]:
from pathlib import Path
print("1. กำลังโหลดข้อมูล...")
# หาไฟล์แบบยืดหยุ่น (โฟลเดอร์ปัจจุบัน, parent, data/, และค้นหาแบบ recursive)
candidates = [
    Path.cwd() / "data" / FILE_1,
    Path.cwd().parent / "data" / FILE_1,
]

file1_path = next((p for p in candidates if p.exists()), None)
if file1_path is None:
    matches = list(Path.cwd().rglob(FILE_1))
    if not matches:
        raise FileNotFoundError(
            f"ไม่พบไฟล์ '{FILE_1}' จากโฟลเดอร์ปัจจุบัน: {Path.cwd()}"
        )
    file1_path = matches[0]

df1 = pd.read_csv(file1_path, low_memory=False)
# โหลดไฟล์ที่ 2 แบบยืดหยุ่นเหมือนกัน
candidates2 = [
    Path.cwd() / "data" / FILE_2,
    Path.cwd().parent / "data" / FILE_2,
]

file2_path = next((p for p in candidates2 if p.exists()), None)
if file2_path is None:
    matches2 = list(Path.cwd().rglob(FILE_2))
    if not matches2:
        raise FileNotFoundError(
            f"ไม่พบไฟล์ '{FILE_2}' จากโฟลเดอร์ปัจจุบัน: {Path.cwd()}"
        )
    file2_path = matches2[0]

df2 = pd.read_csv(file2_path, low_memory=False)
print(f"   -> โหลดสำเร็จ! df1: {df1.shape}, df2: {df2.shape}")

1. กำลังโหลดข้อมูล...
   -> โหลดสำเร็จ! df1: (41361, 62), df2: (500, 9)


In [39]:
print("2. จัดการตัวอักษรขยะ ('nd', 'NA') และแปลงประเภทข้อมูล...")
# เปลี่ยนพวก nd, NA, ช่องว่าง ให้กลายเป็น np.nan
df1 = df1.replace(['nd', 'ND', 'na', 'NA', 'None', 'none', ''], np.nan)
df2 = df2.replace(['nd', 'ND', 'na', 'NA', 'None', 'none', ''], np.nan)

# รายชื่อคอลัมน์ใน df1 ที่ควรเป็นตัวเลข
numeric_cols_df1 = [
    "Latitude_Degrees", "Longitude_Degrees", "Distance_to_Shore", "Turbidity", "Cyclone_Frequency", 
    "Date_Day", "Date_Month", "Date_Year", "Depth_m", "Percent_Cover", "Percent_Bleaching", 
    "ClimSST", "Temperature_Kelvin", "Temperature_Mean", "Temperature_Minimum", "Temperature_Maximum", 
    "Temperature_Kelvin_Standard_Deviation", "Windspeed", "SSTA", "SSTA_Standard_Deviation", 
    "SSTA_Mean", "SSTA_Minimum", "SSTA_Maximum", "SSTA_Frequency", "SSTA_Frequency_Standard_Deviation", 
    "SSTA_FrequencyMax", "SSTA_FrequencyMean", "SSTA_DHW", "SSTA_DHW_Standard_Deviation", "SSTA_DHWMax", 
    "SSTA_DHWMean", "TSA", "TSA_Standard_Deviation", "TSA_Minimum", "TSA_Maximum", "TSA_Mean", 
    "TSA_Frequency", "TSA_Frequency_Standard_Deviation", "TSA_FrequencyMax", "TSA_FrequencyMean", 
    "TSA_DHW", "TSA_DHW_Standard_Deviation", "TSA_DHWMax", "TSA_DHWMean"
]

# บังคับแปลงเป็นตัวเลข (อะไรที่แปลงไม่ได้จะกลายเป็น NaN)
for col in numeric_cols_df1:
    if col in df1.columns:
        df1[col] = pd.to_numeric(df1[col], errors='coerce')

# แปลงคอลัมน์ Date
df1["Date"] = pd.to_datetime(df1["Date"], errors="coerce")
df2["Date"] = pd.to_datetime(df2["Date"], errors="coerce")

# แปลง Heatwave ใน df2 เป็นเลข 0 หรือ 1
if "Marine Heatwave" in df2.columns:
    df2["Marine Heatwave"] = df2["Marine Heatwave"].astype(str).str.lower().map(
        {"true": 1, "false": 0, "1": 1, "0": 0, "yes": 1, "no": 0}
    )

2. จัดการตัวอักษรขยะ ('nd', 'NA') และแปลงประเภทข้อมูล...


In [40]:
print("3. กรองเฉพาะข้อมูลที่มีค่า Percent_Bleaching...")
rows_before = len(df1)
df1 = df1.dropna(subset=["Percent_Bleaching"]).copy()
print(f"   -> ลบข้อมูลที่ไม่มี Target ไป: {rows_before - len(df1):,} แถว")

3. กรองเฉพาะข้อมูลที่มีค่า Percent_Bleaching...
   -> ลบข้อมูลที่ไม่มี Target ไป: 6,846 แถว


In [41]:
print("4. สร้างฟีเจอร์ใหม่ (เช่น แปลงอุณหภูมิ, เช็คเขตศูนย์สูตร)...")
df1["Temperature_C"] = df1["Temperature_Kelvin"] - 273.15
df1["Temp_Range_C"] = (df1["Temperature_Maximum"] - 273.15) - (df1["Temperature_Minimum"] - 273.15)
df1["Is_Tropical"] = (df1["Latitude_Degrees"].abs() <= 23.5).astype(int)

4. สร้างฟีเจอร์ใหม่ (เช่น แปลงอุณหภูมิ, เช็คเขตศูนย์สูตร)...


In [42]:
print("5. ผสานข้อมูลจาก Dataset 2 ตามพิกัดทางภูมิศาสตร์...")
# ลบแถวใน df2 ที่ไม่มีพิกัดหรืออุณหภูมิออกไปก่อน
df2_clean = df2.dropna(subset=["Latitude", "Longitude", "SST (°C)"]).copy()

# สร้างโมเดล BallTree สำหรับคำนวณระยะทางบนพื้นโลก
coords1 = np.radians(df1[["Latitude_Degrees", "Longitude_Degrees"]].values)
coords2 = np.radians(df2_clean[["Latitude", "Longitude"]].values)
tree = BallTree(coords2, metric="haversine")

# หาจุดที่ใกล้ที่สุด (k=1)
dist, idx = tree.query(coords1, k=1)
dist_km = dist.flatten() * 6371.0 # รัศมีโลก

# กรองเอาระยะทางที่ไม่เกิน 500 กิโลเมตร
match_mask = dist_km <= 500

# ดึงข้อมูลมาใส่ df1
df1["Aux_Marine_Heatwave"] = np.where(match_mask, df2_clean["Marine Heatwave"].values[idx.flatten()], 0)
df1["Aux_pH_Level"] = np.where(match_mask, df2_clean["pH Level"].values[idx.flatten()], np.nan)
print(f"   -> จับคู่พิกัดสำเร็จ {match_mask.sum():,} แถว (ในรัศมี 500 กม.)")

5. ผสานข้อมูลจาก Dataset 2 ตามพิกัดทางภูมิศาสตร์...
   -> จับคู่พิกัดสำเร็จ 4,648 แถว (ในรัศมี 500 กม.)


In [43]:
print("6. ลบคอลัมน์ที่เป็นชื่อเฉพาะ และคอมเมนต์ยาวๆ...")
cols_to_drop = [
    "Site_ID", "Sample_ID", "Data_Source", "Reef_ID", "Site_Name", 
    "City_Town_Name", "State_Island_Province_Name", "Date",
    "Site_Comments", "Sample_Comments", "Bleaching_Comments", 
    "Bleaching_Level", "Percent_Cover", "Temperature_Kelvin"
]
df1 = df1.drop(columns=[c for c in cols_to_drop if c in df1.columns])

6. ลบคอลัมน์ที่เป็นชื่อเฉพาะ และคอมเมนต์ยาวๆ...


In [44]:
print("7. จัดการ Outliers (ตัดขอบ 0.5% บน-ล่าง)...")
# หาเฉพาะคอลัมน์ที่เป็นตัวเลข
numeric_features = df1.select_dtypes(include=[np.number]).columns.tolist()
# ยกเว้นคอลัมน์ Target และพวก 0/1 
features_to_clip = [c for c in numeric_features if c not in ["Percent_Bleaching", "Is_Tropical", "Aux_Marine_Heatwave"]]

lower_bounds = df1[features_to_clip].quantile(0.005)
upper_bounds = df1[features_to_clip].quantile(0.995)
df1[features_to_clip] = df1[features_to_clip].clip(lower=lower_bounds, upper=upper_bounds, axis=1)

7. จัดการ Outliers (ตัดขอบ 0.5% บน-ล่าง)...


In [45]:
print("8. เติมค่าว่างด้วยค่ามัธยฐาน แยกตามภูมิภาค (Realm_Name)...")
for col in numeric_features:
    # ถ้าคอลัมน์นั้นมีค่าว่าง
    if df1[col].isnull().any():
        # ถ้ามีคอลัมน์ Realm_Name ให้เติมด้วยค่าของ Realm นั้นก่อน
        if "Realm_Name" in df1.columns:
            df1[col] = df1.groupby("Realm_Name")[col].transform(lambda x: x.fillna(x.median()))
        
        # ถ้าบาง Realm ว่างทั้งหมด หรือไม่มี Realm_Name ให้เติมด้วยค่ามัธยฐานของโลก (สำรอง)
        df1[col] = df1[col].fillna(df1[col].median())

8. เติมค่าว่างด้วยค่ามัธยฐาน แยกตามภูมิภาค (Realm_Name)...


In [46]:
print("9. แปลงคอลัมน์ข้อความ (Categorical) เป็นตัวเลข (One-Hot)...")
categorical_cols = ["Ocean_Name", "Realm_Name", "Exposure", "Substrate_Name"]
# เลือกเฉพาะคอลัมน์ที่ยังมีอยู่ใน df
cat_cols_present = [c for c in categorical_cols if c in df1.columns]

# ทำ One-Hot Encoding
df1 = pd.get_dummies(df1, columns=cat_cols_present, dummy_na=False, drop_first=True)

# เคลียร์คอลัมน์ข้อความที่อาจหลงเหลืออยู่ออกไป เพื่อให้ AI ไม่ Error
object_cols = df1.select_dtypes(include=["object"]).columns.tolist()
if object_cols:
    df1 = df1.drop(columns=object_cols)

9. แปลงคอลัมน์ข้อความ (Categorical) เป็นตัวเลข (One-Hot)...


In [47]:
print("10. ส่งออกไฟล์ข้อมูลพร้อมเทรน...")
df1.to_csv(OUTPUT_FILE, index=False)

print("="*50)
print(f"✅ เสร็จสมบูรณ์! ข้อมูลบันทึกไว้ที่ '{OUTPUT_FILE}'")
print(f"📊 ขนาดข้อมูลที่พร้อมเทรน: {df1.shape[0]:,} แถว | {df1.shape[1]:,} คอลัมน์")
print("="*50)

10. ส่งออกไฟล์ข้อมูลพร้อมเทรน...
✅ เสร็จสมบูรณ์! ข้อมูลบันทึกไว้ที่ 'c:\Users\Jirayut Pimmuen\OneDrive\Desktop\folder for learn\ProjectAI\coral-bleaching-risk-prediction\data\bleaching_model_ready.csv'
📊 ขนาดข้อมูลที่พร้อมเทรน: 34,515 แถว | 62 คอลัมน์
